In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import numpy as np
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from sorl.sorl_wrapper import SorlModelWrapper

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model + checkpoint
model_name = "Qwen/Qwen2.5-0.5B"
model = SorlModelWrapper.from_pretrained(model_name, abstract_vocab_size_list=[128])

hf_repo_id = "Ksgk-fy/sorl_pt"
hf_filename = "qwen2.5-0.5B_gsm8k_K4_v128_i2/final.pt"
ckpt_path = hf_hub_download(repo_id=hf_repo_id, filename=hf_filename)
ckpt = torch.load(ckpt_path, map_location=device)
state_dict = ckpt["model"] if "model" in ckpt else ckpt
clean_sd = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(clean_sd)
model = model.to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)
K = 4

print(f"Device: {device} | Step: {ckpt.get('step', 'N/A')} | Epoch: {ckpt.get('epoch', 'N/A')}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Device: cpu | Step: 2805 | Epoch: 3


In [9]:
# Load GSM8K validation set
import importlib, sorl.cot_utils
importlib.reload(sorl.cot_utils)
from data.pt_dataset import get_dataset
from sorl.cot_utils import generate_with_alignment, probe_answer_logprob, find_collapse_position, render_inline_html
from IPython.display import HTML

val_ds = get_dataset("gsm8k", split="test", tokenizer=tokenizer, max_length=512)
extract_fn = val_ds.extract_answer
print(f"Val set: {len(val_ds)} samples")

Val set: 1319 samples


In [3]:
# Generate + visualize aligned CoT with abstract tokens
NUM_SAMPLES = 5
MAX_NEW_TOKENS = 256

results = []
for i in range(NUM_SAMPLES):
    r = generate_with_alignment(model, tokenizer, val_ds, i, extract_fn, device,
                                max_new_tokens=MAX_NEW_TOKENS, K=K)
    results.append(r)
    status = "OK" if r["correct"] else "X"
    print(f"[{i}] {status} gold={r['gold']} pred={r['pred']} "
          f"abs_reasoning={r['n_abs_reasoning']} abs_answer={r['n_abs_answer']}")

html_out = "".join(render_inline_html(r, i, K=K) for i, r in enumerate(results))
display(HTML(html_out))

[0] OK gold=18 pred=18 abs_reasoning=29 abs_answer=23
[1] X gold=3 pred=6 abs_reasoning=17 abs_answer=35
[2] X gold=70000 pred=62000 abs_reasoning=39 abs_answer=13
[3] OK gold=540 pred=540 abs_reasoning=14 abs_answer=38
[4] X gold=20 pred=40 abs_reasoning=20 abs_answer=32


In [4]:
# Probe answer log-prob at various reasoning chain fractions
PROBE_FRACS = [0.0, 0.1, 0.25, 0.5, 0.75]
NUM_PROBE = 20
base_vocab = model.vocab_sizes[0].item()

all_probe_results = []
for i in range(NUM_PROBE):
    item = val_ds[i]
    input_ids = item["input_ids"].unsqueeze(0).to(device)
    prompt_len = item["prompt_len"]

    ref_ids = item["input_ids"][item["input_ids"] < base_vocab]
    gold = extract_fn(tokenizer.decode(ref_ids, skip_special_tokens=True))
    if gold is None:
        continue

    generated = model.generate(
        input_ids=input_ids[:, :prompt_len],
        max_new_tokens=MAX_NEW_TOKENS, temperature=0.0, K=K,
    )

    collapse_pos = find_collapse_position(generated, prompt_len, base_vocab)
    probe_res, reasoning_len = probe_answer_logprob(
        model, tokenizer, generated, prompt_len, gold,
        probe_fractions=PROBE_FRACS, collapse_pos=collapse_pos,
    )

    traj = generated[0][generated[0] < base_vocab]
    pred = extract_fn(tokenizer.decode(traj, skip_special_tokens=True))
    correct = pred is not None and pred.strip() == gold.strip()

    all_probe_results.append({
        "idx": i, "gold": gold, "pred": pred, "correct": correct,
        "probes": probe_res, "reasoning_len": reasoning_len,
        "collapse_frac": (collapse_pos - prompt_len) / max(reasoning_len, 1)
                         if collapse_pos else None,
    })

    if i < 3:
        print(f"\nSample {i} (gold={gold}, pred={pred}, {'OK' if correct else 'X'}):")
        for label, lp, pos in probe_res:
            print(f"  {label:>10s}: avg_logprob = {lp:.3f}")

print(f"\nProbed {len(all_probe_results)} samples")


Sample 0 (gold=18, pred=18, OK):
          0%: avg_logprob = -5.272
    collapse: avg_logprob = -3.067
         10%: avg_logprob = -5.422
         25%: avg_logprob = -4.742
         50%: avg_logprob = -3.515
         75%: avg_logprob = -4.189
        100%: avg_logprob = -3.750

Sample 1 (gold=3, pred=6, X):
          0%: avg_logprob = -7.672
    collapse: avg_logprob = -7.914
         10%: avg_logprob = -6.207
         25%: avg_logprob = -5.605
         50%: avg_logprob = -8.562
         75%: avg_logprob = -9.789
        100%: avg_logprob = -6.507

Sample 2 (gold=70000, pred=62000, X):
          0%: avg_logprob = -4.134
    collapse: avg_logprob = -4.094
         10%: avg_logprob = -4.870
         25%: avg_logprob = -5.222
         50%: avg_logprob = -2.611
         75%: avg_logprob = -4.968
        100%: avg_logprob = -4.243


KeyboardInterrupt: 

In [15]:
# Abstract CoT analysis from cached results (no re-generation needed)
# Works with both old results (no generated_seq) and new results (with generated_seq)

for i, r in enumerate(results):
    reasoning = r["reasoning_ids"]
    answer = r["answer_ids"]
    all_ids = reasoning + answer

    # Collapse detection: first position where all subsequent abs tokens are the same ID
    collapse_idx = None
    for j in range(len(all_ids)):
        if len(set(all_ids[j:])) == 1:
            collapse_idx = j
            break

    # Compute what abstract CoT would look like
    n_pre_collapse = collapse_idx if collapse_idx is not None else len(all_ids)
    n_post_collapse_abs = len(all_ids) - n_pre_collapse
    # In periodic generation: each abs token has K traj tokens before it
    n_traj_total = len(r["traj_text"].split())  # rough word count
    n_traj_pre = int(n_pre_collapse * K)  # traj tokens kept (approx)
    n_traj_post = max(n_traj_total - n_traj_pre, 0)  # traj tokens dropped

    status = "OK" if r["correct"] else "X"
    collapse_frac = f"{collapse_idx}/{len(all_ids)}" if collapse_idx is not None else "none"

    print(f"[{i}] {status:>2s} gold={r['gold']:>6s} pred={r['pred']:>6s} | "
          f"collapse@{collapse_frac:>8s} | "
          f"unique_reasoning={len(set(reasoning)):>2d}/{len(reasoning):>2d} "
          f"unique_answer={len(set(answer)):>2d}/{len(answer):>2d} | "
          f"~traj_drop={n_traj_post:>3d}")

print("\n--- Abstract CoT pattern (R=reasoning, A=answer, *=diverse, .=collapsed) ---")
for i, r in enumerate(results):
    all_ids = r["reasoning_ids"] + r["answer_ids"]
    n_r = len(r["reasoning_ids"])
    pattern = []
    for j, aid in enumerate(all_ids):
        seg = "R" if j < n_r else "A"
        # Check if this is part of collapsed region
        if len(set(all_ids[j:])) == 1:
            pattern.append(".")
        else:
            pattern.append("*" if seg == "R" else "^")
    print(f"  [{i}] {''.join(pattern)}")

[0] OK gold=    18 pred=    18 | collapse@    3/52 | unique_reasoning= 2/29 unique_answer= 1/23 | ~traj_drop= 91
[1]  X gold=     3 pred=     6 | collapse@    1/52 | unique_reasoning= 2/17 unique_answer= 1/35 | ~traj_drop=100
[2]  X gold= 70000 pred= 62000 | collapse@    2/52 | unique_reasoning= 3/39 unique_answer= 1/13 | ~traj_drop= 28
[3] OK gold=   540 pred=   540 | collapse@    1/52 | unique_reasoning= 2/14 unique_answer= 1/38 | ~traj_drop= 74
[4]  X gold=    20 pred=    40 | collapse@    3/52 | unique_reasoning= 3/20 unique_answer= 1/32 | ~traj_drop=116

--- Abstract CoT pattern (R=reasoning, A=answer, *=diverse, .=collapsed) ---
  [0] ***.................................................
  [1] *...................................................
  [2] **..................................................
  [3] *...................................................
  [4] ***.................................................


In [1]:
# Build inner-monologue response: [abstract reasoning] + [#### answer]
import importlib, sorl.cot_utils
importlib.reload(sorl.cot_utils)
from sorl.cot_utils import build_inner_monologue_response

base_vocab = model.vocab_sizes[0].item()

for i, r in enumerate(results):
    im = build_inner_monologue_response(r, tokenizer, base_vocab)
    status = "OK" if im["correct"] else "X"

    # Decode the abstract tokens as placeholder text for display
    abs_display = " ".join([f"[A{rid}]" for rid in r["reasoning_ids"]])

    print(f"\n[{i}] {status} gold={im['gold']} pred={im['pred']}")
    print(f"  Original reasoning: {im['original_n_traj_words']} words")
    print(f"  Inner monologue:    {im['n_abs']} abstract tokens + {im['n_answer_tokens']} answer tokens")
    print(f"  Response: {abs_display} {im['answer_text']}")

NameError: name 'model' is not defined